# 01 · Understanding the data: what does a microservice failure look like?

**Goal of this project:** when something breaks in a system made of many small services, find **which service caused it** (root-cause *localisation*) and **what kind of fault** it is. This is the "root-cause analysis" step of incident management, done on real telemetry.

### The dataset: RCAEval, suite RE2
RCAEval (WWW 2025 benchmark) ran three open-source demo shops/apps, deliberately broke one service at a time and recorded everything:

| System | Short name | Services with metrics | What it is |
|---|---|---|---|
| Online Boutique | `ob` | 12 | Google's e-commerce demo (Go, Python, Java, …) |
| Sock Shop | `ss` | 15 | Weaveworks' sock shop |
| Train Ticket | `tt` | 68 | a large ticket-booking system (Java) |

**Fault types (6):** CPU stress, memory stress, disk stress, network delay, packet loss, socket stress.

**One case** = 12 minutes of normal operation → fault injected at `inject_time` → 12 more minutes. We get:
- **metrics** (1 row per second; CPU, memory, disk I/O, sockets, workload, errors, latency per service),
- **logs** (every log line with its container),
- **traces** (every request "span": which service, which operation, how long it took). Not recorded for Sock Shop.

The **label** (`root_cause_service`) is known because the fault was injected on purpose, a real ground truth, unlike most incident datasets.

> **Why this matters:** in a bank's operations centre an alert says "checkout is slow", but the cause is often a *different* service. Automatically ranking the likely culprits saves the on-call engineer time.

All data work in notebooks 01–02 uses only **pandas, numpy, matplotlib and seaborn**.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid"); pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
ROOT = Path.cwd().parent                                  # the project folder (notebooks run inside notebooks/)
RAW, PROCESSED, FIG = ROOT / "data" / "raw", ROOT / "data" / "processed", ROOT / "reports" / "figures"
PROCESSED.mkdir(parents=True, exist_ok=True); FIG.mkdir(parents=True, exist_ok=True)

cases = pd.read_parquet(RAW / "cases.parquet")                     # the benchmark's index: one row per case
cases = cases[cases["suite"] == "RE2"].sort_values("case").reset_index(drop=True)
print(len(cases), "cases")
cases[["case", "system", "root_cause_service", "fault", "repetition", "inject_time"]].head()

In [ ]:
cases.groupby("system").agg(cases=("case", "size"), root_cause_services=("root_cause_service", "nunique"),
                            fault_types=("fault", "nunique"), metrics=("n_metrics", "median"),
                            log_lines=("n_logs", "median"), spans=("n_traces", "median"))

**Reading the table:** 90 cases per system = 5 root-cause services × 6 fault types × 3 repetitions. Train Ticket is much bigger (360+ metric columns, ~1M spans per case). Sock Shop has no traces (0 spans).

In [ ]:
pd.crosstab([cases["system"], cases["root_cause_service"]], cases["fault"])

Every (service, fault) pair appears exactly 3 times. **Important consequence for evaluation:** the 3 repetitions are near-copies of each other. If one repetition is in training and another in test, the model can "recognise" the case instead of reasoning about it. So our splits always keep all 3 repetitions together (notebook 04).

## One case up close: a CPU fault on `checkoutservice`
Each case folder has `metrics.parquet`, `logs.parquet` and (except Sock Shop) `traces.parquet`. The metrics file is **wide**: one row per second, one column per `<service>_<metric>`.

In [ ]:
example = cases[cases["case"] == "re2ob_checkoutservice_cpu_1"].iloc[0]
wide = pd.read_parquet(RAW / example["case"] / "metrics.parquet")
print(wide.shape)
wide.iloc[:3, :6]

Analysis is easier in **long** format: one row per (second, service, metric).
- `melt` stacks all the metric columns into two columns: the column name and the value.
- `str.rsplit("_", n=1)` splits `checkoutservice_cpu` at the **last** underscore into service and metric (metric names like `latency-50` contain no underscore).

In [ ]:
long = wide.melt(id_vars="time", var_name="column", value_name="value")
long[["service", "metric"]] = long["column"].str.rsplit("_", n=1, expand=True)
long["minute"] = (long["time"] - example["inject_time"]) / 60          # 0 = the moment the fault is injected
print(long.shape, "| services:", long["service"].nunique(), "| metric types:", sorted(long["metric"].unique()))
long.head()

## What the model must choose from
For each case the candidate list is **every service that has metrics**, not just the 5 that the benchmark happens to break (using that list would be cheating: in real life you don't know which 5 services will fail).

In [ ]:
def n_services(case):
    columns = pd.read_parquet(RAW / case / "metrics.parquet").columns.drop("time")
    return columns.str.rsplit("_", n=1).str[0].nunique()

n = cases.groupby("system")["case"].first().map(n_services)          # one case per system is enough
pd.DataFrame({"candidates": n, "random top-1": 1 / n, "random top-3": 3 / n}).round(3)

A random guess finds the culprit first time 8% (Online Boutique), 7% (Sock Shop) or **1.5%** (Train Ticket) of the time. That's the floor.

## What does the failure look like?
Each line is one service; the root cause is red.

In [ ]:
long["role"] = np.where(long["service"] == example["root_cause_service"], "root cause", "other services")
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6), sharex=True)
for ax, metric in zip(axes, ["cpu", "mem", "latency-90"]):
    sns.lineplot(data=long[long["metric"] == metric], x="minute", y="value", units="service", estimator=None,
                 hue="role", hue_order=["other services", "root cause"], palette=["#9aa7b4", "#c0392b"],
                 size="role", sizes={"other services": 0.7, "root cause": 2.2}, legend=(metric == "cpu"), ax=ax)
    ax.axvline(0, color="k", ls="--", lw=1)
    ax.set_title(metric); ax.set_xlabel("minutes from fault injection"); ax.set_ylabel("")
plt.tight_layout(); plt.savefig(FIG / "example_cpu_fault.png", dpi=120); plt.show()

The red line (the root cause) jumps in CPU right at the injection (dashed line). Other services barely move. **But it is not always this clean:**
- a *delay* or *loss* fault on one service shows up as **higher latency in its callers** too (the slowness travels upstream),
- some metrics are noisy or flat (a service that normally has 0 errors makes any error look "infinitely" anomalous),
- Train Ticket has 68 services, so there are many chances for an innocent service to look odd.

That is why we compute several kinds of evidence per service and let a model learn how to weigh them.

## Logs and traces around the injection

In [ ]:
logs = pd.read_parquet(RAW / example["case"] / "logs.parquet")
logs["minute"] = (logs["timestamp"] - example["inject_time"]) // 60
per_minute = logs.groupby(["minute", "container_name"]).size().unstack(fill_value=0)
busiest = per_minute.sum().nlargest(6).index                                # the 6 containers that log the most
ax = per_minute[busiest].plot(figsize=(10, 3.4), lw=1.2)
ax.axvline(0, color="k", ls="--"); ax.set_ylabel("log lines / minute"); ax.set_title("log volume per container")
plt.tight_layout(); plt.show()

In [ ]:
traces_file = RAW / example["case"] / "traces.parquet"
if traces_file.exists():
    traces = pd.read_parquet(traces_file, columns=["serviceName", "startTimeMillis", "duration"])
    traces["minute"] = (traces["startTimeMillis"] // 1000 - example["inject_time"]) // 60
    p95 = traces.groupby(["minute", "serviceName"])["duration"].quantile(0.95).unstack() / 1000   # microseconds -> ms
    ax = p95.plot(figsize=(10, 3.4), lw=1.2, logy=True)
    ax.axvline(0, color="k", ls="--"); ax.set_ylabel("p95 span duration (ms)"); ax.set_title("trace latency per service")
    plt.tight_layout(); plt.show()

### Takeaways
1. Each case is a **ranking problem**: order 12–68 candidate services so the true culprit comes first.
2. Evidence comes from three sources (metrics, logs, traces). Traces are missing for Sock Shop, so models must cope with missing data.
3. The 3 repetitions per (service, fault) must stay together in any train/test split.

➡️ Next: **02_feature_engineering**: turn each service's behaviour before/after the fault into numbers.